# 01. Data Preparation and Quality Rules

This notebook documents the data-preparation stage of the MSc dissertation **Using Sentiment Analysis to Improve Student Support Services**.

The focus is not only technical cleaning. From a Business Analyst perspective, each rule defines which feedback is considered reliable enough to support downstream reporting and service decisions.

> **Data note:** The original university feedback is not published. `sample_feedback.csv` contains synthetic examples only.


## Business rule: define analysis-ready feedback

The final workflow removed missing or blank comments, duplicate responses, and comments shorter than three words. Text was standardized while preserving sentence context for transformer-based modeling.

**Why this matters:** inconsistent inclusion rules can distort sentiment rates and make semester-to-semester comparisons unreliable. The final analytical dataset contained **1,022 comments** from an original 1,178 responses.


In [ ]:
from pathlib import Path
import re
import pandas as pd

DATA_PATH = Path('../data/sample_feedback.csv')
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Business rule 1: exclude records with no usable feedback
df_cleaned = df.dropna(subset=['comment']).copy()
df_cleaned = df_cleaned[df_cleaned['comment'].str.strip().ne('')]

# Business rule 2: avoid double-counting repeated feedback
df_cleaned = df_cleaned.drop_duplicates(subset=['comment']).copy()

# Business rule 3: remove responses too short to provide interpretable evidence
df_cleaned['word_count'] = df_cleaned['comment'].str.split().str.len()
df_cleaned = df_cleaned[df_cleaned['word_count'] >= 3].copy()


In [ ]:
def preprocess_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'[^\w\s.,!?-]', ' ', text)
    return ' '.join(text.split())

df_cleaned['comment_preprocessed'] = df_cleaned['comment'].apply(preprocess_text)
df_cleaned[['comment', 'comment_preprocessed']].head()

## Data-quality validation from the dissertation

The original research reported:

- Original responses: **1,178**
- Final analysis-ready responses: **1,022**
- Data retention rate: **86.76%**
- Missing/blank responses removed: **9**
- Duplicate comments removed: **81**
- Short responses (<3 words) removed: **147**

**BA interpretation:** these figures create an auditable bridge between raw input and the population used for reporting. The synthetic sample below demonstrates the checks but is not expected to reproduce the dissertation counts.


In [ ]:
# Basic quality checks for the public sample
quality_summary = {
    'rows': len(df_cleaned),
    'unique_comments': df_cleaned['comment_preprocessed'].nunique(),
    'avg_words_per_comment': round(df_cleaned['word_count'].mean(), 2),
}
quality_summary